In [4]:
pip install matplotlib

  Using cached matplotlib-3.10.8-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (117 kB)
  Using cached kiwisolver-1.5.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
  Using cached pillow-12.1.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.10.8-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (8.7 MB)
Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (362 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.62.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17

In [1]:
import os
import gc
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from itertools import combinations
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.interpolate import interp1d
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K

# ════════════════════════════════════════════════════════
#  SETTINGS
# ════════════════════════════════════════════════════════
MAX_MIX_COMBOS     = 2000    # max 3-molecule mixtures to generate
MIX_POINTS         = 1000    # interpolation resolution for mixing
WEIGHTS_3          = [1/3, 1/3, 1/3]    # equal-weight ternary mixture

LATENT_DIM         = 64
EPOCHS             = 150
BATCH_SIZE         = 64
DPI                = 300
PREVIEW_N          = 24
DISPLAY_TRANSMITTANCE = False   # False = absorbance (peaks UP)

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# ════════════════════════════════════════════════════════
#  PATHS
# ════════════════════════════════════════════════════════
INPUT_DIR   = os.path.expanduser('~/Downloads/Test Astro Molecules')
OUTPUT_MIX  = os.path.join(INPUT_DIR, 'mixed_spectra_output_3molecules_a')
FIGURES_DIR = os.path.join(OUTPUT_MIX,  'figures_ternary')
os.makedirs(OUTPUT_MIX,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"  INPUT_DIR   = {INPUT_DIR}")
print(f"  OUTPUT_MIX  = {OUTPUT_MIX}")
print(f"  FIGURES_DIR = {FIGURES_DIR}\n")

# ════════════════════════════════════════════════════════
#  JOURNAL STYLE — NO GRID — EVERYTHING BOLD
# ════════════════════════════════════════════════════════
plt.rcParams.update({
    'figure.facecolor':  'white', 'axes.facecolor':  'white',
    'axes.edgecolor':    '#222222', 'axes.linewidth': 1.1,
    'axes.grid':         False,
    'axes.labelcolor':   '#111111', 'axes.titlecolor':'#111111',
    'axes.titlesize':    11,  'axes.labelsize':  10,
    'axes.titleweight':  'bold', 'axes.labelweight': 'bold',
    'xtick.direction':   'in', 'ytick.direction': 'in',
    'xtick.major.size':  3.5,  'ytick.major.size':3.5,
    'xtick.major.width':  1.1, 'ytick.major.width':1.1,
    'xtick.color':       '#222222', 'ytick.color':   '#222222',
    'xtick.labelsize':   9, 'ytick.labelsize': 9,
    'text.color':        '#111111',
    'legend.facecolor':  'white', 'legend.edgecolor':'#aaaaaa',
    'legend.fontsize':   8.5, 'legend.framealpha':0.92,
    'lines.linewidth':   1.5,
    'font.family':       'sans-serif',
    'font.sans-serif':   ['DejaVu Sans', 'Arial'],
    'font.size':         10,
    'font.weight':       'bold',        # <-- makes ALL text bold by default
    'figure.titleweight':'bold',
    'savefig.facecolor': 'white', 'savefig.edgecolor':'none',
    'figure.max_open_warning': 5,
    'mathtext.default':  'bf',          # bold math/number glyphs too
})

TRAIN_COL = '#1f77b4'; VAL_COL = '#d62728'
GOOD_COL  = '#2ca02c'; FAIR_COL = '#ff7f0e'; POOR_COL = '#d62728'
ACCENT    = '#9467bd'; RES_COL  = '#e07b00'

# ── Helper: force bold on every tick label of an axes ──
def bold_ticks(ax):
    """Explicitly bold every x/y tick label (belt-and-braces on top of rcParams)."""
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight('bold')

# ── Display helpers ───────────────────────────────────
def _yd(arr):
    return 1.0 - arr if DISPLAY_TRANSMITTANCE else arr
def _ylabel():
    return 'Transmittance [0–1]' if DISPLAY_TRANSMITTANCE else 'Absorbance'
def _ylim():
    return (1.12, -0.04) if DISPLAY_TRANSMITTANCE else (-0.04, 1.12)
def _yticks():
    return [1.0,0.8,0.6,0.4,0.2,0.0] if DISPLAY_TRANSMITTANCE \
           else [0.0,0.2,0.4,0.6,0.8,1.0]

def section_title(t):
    print(f"\n{'═'*60}\n  {t}\n{'═'*60}")

def save_figure(fig, filename):
    # belt-and-braces: bold every text object left on the figure before saving
    for ax in fig.get_axes():
        bold_ticks(ax)
        for txt in ax.texts:
            txt.set_fontweight('bold')
        leg = ax.get_legend()
        if leg is not None:
            for t in leg.get_texts():
                t.set_fontweight('bold')
            if leg.get_title() is not None:
                leg.get_title().set_fontweight('bold')
    for txt in fig.texts:
        txt.set_fontweight('bold')
    for ext in ['png','pdf']:
        path = os.path.join(FIGURES_DIR, f"{filename}.{ext}")
        fig.savefig(path, dpi=DPI, bbox_inches='tight',
                    facecolor='white', edgecolor='none')
        print(f"  ✓ Saved → {path}")
    plt.close(fig); gc.collect()

# ════════════════════════════════════════════════════════
#  PAH SKIP LIST
# ════════════════════════════════════════════════════════
PAH_SKIP_KEYWORDS = [
    'pah','naphthalene','anthracene','pyrene','fluorene','phenanthrene',
    'chrysene','acenaphthylene','acenaphthene','fluoranthene','benzo',
    'perylene','indene','coronene','triphenylene',
    'pentacene','tetracene','naphthacene','hexacene','heptacene',
    'acene','rubrene','ovalene','terylene','quaterrylene',
    'picene','dibenzo','benzopyrene',
    'coronylene',
    'dicoronylene',
    'corannulene',
    'sumanene',
    'helicene',
    'circulene',
    'kekulene',
]
def is_pah_file(fn):
    fn_lower = fn.lower()
    if any(kw in fn_lower for kw in PAH_SKIP_KEYWORDS):
        return True
    stem = fn_lower.replace('.csv','')
    for prefix in ('mix3_','mix2_','mix_'):
        if stem.startswith(prefix):
            stem = stem[len(prefix):]
            break
    tokens = stem.split('_')
    for kw in PAH_SKIP_KEYWORDS:
        for tok in tokens:
            if kw in tok:
                return True
    return False

# ════════════════════════════════════════════════════════
#  TRANSMITTANCE → ABSORBANCE CONVERSION
# ════════════════════════════════════════════════════════
def detect_and_convert(ab, fname=''):
    vals    = ab[np.isfinite(ab)]
    val_max = float(np.max(vals))
    val_min = float(np.min(vals))
    val_med = float(np.median(vals))

    if val_max > 1.5 and val_min >= 0:
        t  = np.clip(ab, 1e-4, 100.0)
        A  = -np.log10(t / 100.0)
        if fname:
            print(f"    [CONVERTED %T→A]  {fname}  "
                  f"raw_max={val_max:.2f}  A_range=[{A.min():.3f},{A.max():.3f}]")
        return A.astype(np.float32), '%T → Absorbance', True

    elif val_max <= 1.5 and val_min >= 0 and val_med > 0.15:
        t  = np.clip(ab, 1e-6, 1.0)
        A  = -np.log10(t)
        if fname:
            print(f"    [CONVERTED T-frac→A]  {fname}  "
                  f"raw_med={val_med:.3f}  A_range=[{A.min():.3f},{A.max():.3f}]")
        return A.astype(np.float32), 'T-frac → Absorbance', True

    else:
        return ab.astype(np.float32), 'Absorbance', False

# ════════════════════════════════════════════════════════
#  PHYSICAL NORMALISATION  →  guaranteed [0, 1]
# ════════════════════════════════════════════════════════
def physical_normalise(s):
    s  = s.astype(np.float64) - s.min()
    pk = s.max()
    if pk > 1e-12:
        s = s / pk
    return s.astype(np.float32)

# ════════════════════════════════════════════════════════
#  RAMP / ARTIFACT DETECTOR
# ════════════════════════════════════════════════════════
RAMP_THRESH = 0.92   # |Pearson r| above this → ramp artifact

def is_ramp_spectrum(s):
    """Return True if spectrum is dominated by a linear monotonic trend."""
    s = np.asarray(s, dtype=np.float64)
    if np.std(s) < 1e-10:
        return True   # flat → also useless
    ramp = np.linspace(0.0, 1.0, len(s))
    c_up = float(np.corrcoef(s, ramp)[0, 1])
    c_dn = float(np.corrcoef(s, ramp[::-1])[0, 1])
    return max(abs(c_up), abs(c_dn)) > RAMP_THRESH

# ════════════════════════════════════════════════════════
#  STEP 1 — LOAD + CONVERT (T→A) ALL INDIVIDUAL SPECTRA
# ════════════════════════════════════════════════════════
section_title("STEP 1 — LOADING & CONVERTING INDIVIDUAL SPECTRA")

raw_spectra  = {}   # fname → {'x': wn_array, 'y': absorbance_array, 'unit': str}
skip_pah     = []
skip_bad     = []
n_converted  = 0
n_already_A  = 0

csv_files_single = [f for f in sorted(os.listdir(INPUT_DIR))
                    if f.lower().endswith('.csv')]

print(f"  Found {len(csv_files_single)} CSV files in {INPUT_DIR}")

for fname in csv_files_single:
    fpath = os.path.join(INPUT_DIR, fname)

    if is_pah_file(fname):
        skip_pah.append(fname); continue

    try:
        df = pd.read_csv(fpath)
    except Exception as e:
        print(f"  [SKIP ERR] {fname}: {e}"); skip_bad.append(fname); continue

    x = pd.to_numeric(df.iloc[:,0], errors='coerce').values
    y = pd.to_numeric(df.iloc[:,1], errors='coerce').values
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 10:
        skip_bad.append(fname); continue

    x = x[valid].astype(np.float32)
    y = y[valid].astype(np.float32)

    y_abs, unit_label, was_cv = detect_and_convert(y, fname)

    y_norm_test = physical_normalise(y_abs)
    if is_ramp_spectrum(y_norm_test):
        print(f"  [SKIP RAMP]  {fname}  (linear/monotonic — not a real IR spectrum)")
        skip_bad.append(fname); continue

    if was_cv: n_converted += 1
    else:      n_already_A += 1

    si = np.argsort(x)
    raw_spectra[fname] = {
        'x':    x[si],
        'y':    y_abs[si],
        'unit': unit_label,
        'converted': was_cv,
    }

print(f"\n  {'─'*55}")
print(f"  ✓ Loaded           : {len(raw_spectra)}")
print(f"  ~ Converted (T→A)  : {n_converted}")
print(f"  ✓ Already absorbance: {n_already_A}")
print(f"  ✗ PAH skipped      : {len(skip_pah)}")
print(f"  ✗ Bad / unreadable : {len(skip_bad)}")
print(f"  {'─'*55}")
print(f"  All individual spectra are now in ABSORBANCE units.")
print(f"  Mixing will be done in absorbance space (Beer-Lambert).\n")

if len(raw_spectra) < 3:
    raise ValueError(
        f"Need at least 3 non-ramp, non-PAH spectra to form a ternary mixture. "
        f"Only {len(raw_spectra)} loaded. Check PAH_SKIP_KEYWORDS and ramp filter.")

section_title("STEP 1b — PURGING STALE PAH-CONTAINING MIXED CSVs")

n_purged = 0
if os.path.isdir(OUTPUT_MIX):
    for root, _, files in os.walk(OUTPUT_MIX):
        for f in files:
            if not f.lower().endswith('.csv'): continue
            if 'figures' in root: continue
            if is_pah_file(f):
                full = os.path.join(root, f)
                os.remove(full)
                print(f"  [PURGED PAH]  {f}")
                n_purged += 1
if n_purged:
    print(f"  ✓ Purged {n_purged} stale PAH-containing mixed CSVs from {OUTPUT_MIX}/")
else:
    print(f"  ✓ No stale PAH CSVs found — output folder is clean")

section_title("STEP 2 — MIXING 3-MOLECULE COMBINATIONS (absorbance space)")

valid_files = list(raw_spectra.keys())
all_combos  = list(combinations(valid_files, 3))
n_possible  = len(all_combos)
n_sample    = min(MAX_MIX_COMBOS, n_possible)
selected    = random.sample(all_combos, n_sample)

print(f"  Total possible 3-combos : {n_possible}")
print(f"  Generating              : {n_sample}")
print(f"  Mix weights             : {WEIGHTS_3}")
print(f"  Interpolation pts/mix   : {MIX_POINTS}\n")

mix3_count = 0
skip_nooverlap = 0

for f1, f2, f3 in selected:
    d1, d2, d3 = raw_spectra[f1], raw_spectra[f2], raw_spectra[f3]

    min_x = max(d1['x'].min(), d2['x'].min(), d3['x'].min())
    max_x = min(d1['x'].max(), d2['x'].max(), d3['x'].max())
    if min_x >= max_x:
        skip_nooverlap += 1; continue

    common_x = np.linspace(min_x, max_x, MIX_POINTS)

    y1 = np.interp(common_x, d1['x'], d1['y'])
    y2 = np.interp(common_x, d2['x'], d2['y'])
    y3 = np.interp(common_x, d3['x'], d3['y'])

    mixed_y = WEIGHTS_3[0]*y1 + WEIGHTS_3[1]*y2 + WEIGHTS_3[2]*y3

    n1 = os.path.splitext(f1)[0]
    n2 = os.path.splitext(f2)[0]
    n3 = os.path.splitext(f3)[0]
    out_name = f"mix3_{n1}_{n2}_{n3}.csv"
    out_path = os.path.join(OUTPUT_MIX, out_name)
    pd.DataFrame({'Wavenumber': common_x, 'Mixed_Absorbance': mixed_y}
                 ).to_csv(out_path, index=False)
    mix3_count += 1

print(f"  ✓ {mix3_count} mixed spectra saved → {OUTPUT_MIX}/")
if skip_nooverlap:
    print(f"  ✗ {skip_nooverlap} combinations skipped (no wavenumber overlap)")

section_title("STEP 3 — LOADING MIXED SPECTRA + PHYSICAL NORMALISE [0,1]")

spectra, names, wavenumbers_list = [], [], []
conversion_log  = []
skipped_pah_mix = []
skipped_bad_mix = []
converted_mix   = []
kept_as_is_mix  = []

csv_mixed = []
for root, _, files in os.walk(OUTPUT_MIX):
    for f in sorted(files):
        if f.lower().endswith('.csv') and 'figures' not in root:
            csv_mixed.append(os.path.join(root, f))

print(f"  Found {len(csv_mixed)} mixed CSV files")

for path in csv_mixed:
    fname = os.path.basename(path)
    base  = fname.replace('.csv','')

    if is_pah_file(fname):
        skipped_pah_mix.append(fname); continue

    try:
        df = pd.read_csv(path, usecols=[0,1], header=0,
                         dtype=np.float32, na_filter=True)
    except Exception as e:
        print(f"  [SKIP ERR] {fname}: {e}")
        skipped_bad_mix.append(fname); continue

    if df.shape[1] < 2 or len(df) < 10:
        skipped_bad_mix.append(fname); del df; continue

    wn = df.iloc[:,0].values.astype(np.float32)
    ab = df.iloc[:,1].values.astype(np.float32)
    del df

    mask = np.isfinite(wn) & np.isfinite(ab)
    wn, ab = wn[mask], ab[mask]
    if len(wn) < 10:
        skipped_bad_mix.append(fname); continue

    ab, unit_label, was_cv = detect_and_convert(ab, '')
    if was_cv: converted_mix.append(fname)
    else:      kept_as_is_mix.append(fname)

    raw_abs_min = float(ab.min())
    raw_abs_max = float(ab.max())
    raw_abs_med = float(np.median(ab))

    ab = physical_normalise(ab)

    if is_ramp_spectrum(ab):
        print(f"  [SKIP RAMP] {fname[:55]}")
        skipped_bad_mix.append(fname)
        if was_cv: converted_mix.pop()
        else:      kept_as_is_mix.pop()
        continue

    si = np.argsort(wn)

    spectra.append(ab[si]); wavenumbers_list.append(wn[si]); names.append(base)
    conversion_log.append({
        'file': base, 'unit': unit_label, 'converted': was_cv,
        'abs_min': raw_abs_min, 'abs_max': raw_abs_max, 'abs_med': raw_abs_med,
    })

print(f"\n  ✓ Loaded          : {len(spectra)}")
print(f"  ✗ PAH skipped     : {len(skipped_pah_mix)}")
print(f"  ✗ Bad/ramp        : {len(skipped_bad_mix)}  (includes ramp artifacts)")
print(f"  ~ Re-converted    : {len(converted_mix)}  (should be 0 — safety check)")
print(f"  ✓ Already absorb. : {len(kept_as_is_mix)}")

if len(spectra) == 0:
    raise ValueError("No mixed spectra loaded.")

# ════════════════════════════════════════════════════════
#  CONVERSION VERIFICATION PLOT (Step 3b)
# ════════════════════════════════════════════════════════
section_title("STEP 3b — CONVERSION VERIFICATION PLOT")

n_pv = min(PREVIEW_N, len(spectra))
vi   = np.linspace(0, len(spectra)-1, n_pv, dtype=int)
nc_v = min(4, n_pv); nr_v = int(np.ceil(n_pv/nc_v))

fig, axes = plt.subplots(nr_v, nc_v, figsize=(5*nc_v, 3.5*nr_v),
                         facecolor='white', squeeze=False)
fig.suptitle(
    f'Unit Conversion Verification  ({n_pv} of {len(spectra)} shown)\n'
    'All spectra should be in absorbance (peaks UP, baseline at 0)\n'
    'Orange = had residual T→A applied  |  Blue = confirmed absorbance',
    fontsize=11, fontweight='bold', y=1.03)

for pi, di in enumerate(vi):
    r, c = divmod(pi, nc_v); ax = axes[r][c]
    lg   = conversion_log[di]
    col  = '#e07b00' if lg['converted'] else TRAIN_COL
    lbl  = 'T→A (safety)' if lg['converted'] else 'Abs ✓'
    ax.plot(wavenumbers_list[di], spectra[di], color=col, lw=1.2, alpha=0.9)
    ax.set_title(f"{lg['file'][:28]}\n{lg['unit']}  [{lbl}]",
                 fontsize=6.5, color=col, fontweight='bold')
    ax.set_xlabel('Wavenumber (cm⁻¹)', fontsize=6.5, fontweight='bold')
    ax.set_ylabel('Norm. Absorbance', fontsize=6.5, fontweight='bold')
    ax.set_ylim(-0.05, 1.10); ax.tick_params(labelsize=5.5)
    bold_ticks(ax)
    ax.text(0.02, 0.97,
            f"max={lg['abs_max']:.3f}\nmed={lg['abs_med']:.3f}",
            transform=ax.transAxes, fontsize=5.5, va='top', fontweight='bold',
            bbox=dict(facecolor='#f5f5f5', edgecolor=col,
                      boxstyle='round,pad=0.3', alpha=0.92))

for pi in range(n_pv, nr_v*nc_v):
    r, c = divmod(pi, nc_v); axes[r][c].set_visible(False)
plt.tight_layout()
save_figure(fig, 'fig0_conversion_verification')
print(f"  All {len(spectra)} spectra should show peaks pointing UP.\n")

# ════════════════════════════════════════════════════════
#  STEP 4 — RESAMPLE → CROP (600–3500 cm⁻¹) → RE-NORMALISE
# ════════════════════════════════════════════════════════
section_title("STEP 4 — RESAMPLE + CROP")

li        = int(np.argmax([len(s) for s in spectra]))
common_wn = wavenumbers_list[li].copy()
tl        = (len(common_wn)//4)*4
common_wn = common_wn[:tl]

X = np.empty((len(spectra), tl), dtype=np.float32)
for i, (sp, ow) in enumerate(zip(spectra, wavenumbers_list)):
    fn   = interp1d(ow, sp, kind='linear', bounds_error=False, fill_value='extrapolate')
    X[i] = fn(common_wn).astype(np.float32)
    if (i+1) % 500 == 0:
        print(f"  Resampled {i+1}/{len(spectra)}...")
del spectra, wavenumbers_list; gc.collect()

WN_MIN, WN_MAX = 600.0, 3500.0
ci = np.where((common_wn >= WN_MIN) & (common_wn <= WN_MAX))[0]
ci = ci[:(len(ci)//4)*4]
common_wn = common_wn[ci]; X = X[:,ci]; tl = len(common_wn)

for i in range(len(X)):
    X[i] = physical_normalise(X[i])
X = np.clip(X, 0.0, 1.0)
print(f"✓ Shape {X.shape}  range [{X.min():.5f}, {X.max():.5f}]")

# ── Raw spectra overview ─────────────────────────────
n_prev = min(PREVIEW_N, len(X)); pidx2 = np.linspace(0,len(X)-1,n_prev,dtype=int)
nc_p=min(4,n_prev); nr_p=int(np.ceil(n_prev/nc_p))
fig,axes=plt.subplots(nr_p,nc_p,figsize=(5*nc_p,3.5*nr_p),facecolor='white',squeeze=False)
fig.suptitle(f'Raw Mixed Ternary Spectra — Absorbance [0,1]  ({n_prev} of {len(X)})',
             fontsize=13,fontweight='bold',y=1.01)
cols_cyc=plt.cm.tab20(np.linspace(0,1,n_prev))
for pi,di in enumerate(pidx2):
    r,c=divmod(pi,nc_p); ax=axes[r][c]; lg=conversion_log[di]
    ax.plot(common_wn,X[di],color=cols_cyc[pi],lw=1.2,alpha=0.9)
    ax.set_title(f"{names[di][:25]}\n{lg['unit']}",fontsize=6.5,fontweight='bold')
    ax.set_xlabel('Wavenumber (cm⁻¹)',fontsize=6.5,fontweight='bold')
    ax.set_ylabel('Absorbance',fontsize=6.5,fontweight='bold')
    ax.set_ylim(-0.04,1.12); ax.tick_params(labelsize=5.5)
    bold_ticks(ax)
for pi in range(n_prev,nr_p*nc_p):
    r,c=divmod(pi,nc_p); axes[r][c].set_visible(False)
plt.tight_layout()
save_figure(fig,'fig1_raw_ternary_spectra')

# ════════════════════════════════════════════════════════
#  STEP 5 — STANDARDISE + SPLIT (75/25)
# ════════════════════════════════════════════════════════
section_title("STEP 5 — STANDARDISE + SPLIT")

X_std, scalers = np.zeros_like(X), []
for i in range(len(X)):
    sc=StandardScaler()
    X_std[i]=sc.fit_transform(X[i].reshape(-1,1)).ravel()
    scalers.append(sc)
X_std3=X_std[...,np.newaxis]

ai=np.arange(len(X))
if len(X)>=4:
    train_idx,val_idx=train_test_split(ai,test_size=0.25,random_state=42)
elif len(X)>=2:
    train_idx=list(ai[:-1]); val_idx=list(ai[-1:])
else:
    train_idx=list(ai); val_idx=list(ai)
train_idx=list(train_idx); val_idx=list(val_idx)
print(f"✓ Train {len(train_idx)} | Val {len(val_idx)}")

# ════════════════════════════════════════════════════════
#  STEP 6 — CNN AUTOENCODER
# ════════════════════════════════════════════════════════
section_title("STEP 6 — CNN AUTOENCODER")

def build_autoencoder(L, latent=64):
    inp=layers.Input(shape=(L,1))
    x=layers.Conv1D(16,7,activation='relu',padding='same')(inp)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Conv1D(32,5,activation='relu',padding='same')(x)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Conv1D(64,3,activation='relu',padding='same')(x)
    x=layers.BatchNormalization()(x)
    sb=x.shape[1:]; x=layers.Flatten()(x)
    z=layers.Dense(latent,activation='relu',name='latent')(x)
    x=layers.Dense(sb[0]*sb[1],activation='relu')(z)
    x=layers.Reshape(sb)(x)
    x=layers.Conv1D(64,3,activation='relu',padding='same')(x)
    x=layers.BatchNormalization()(x); x=layers.UpSampling1D(2)(x)
    x=layers.Conv1D(32,5,activation='relu',padding='same')(x)
    x=layers.BatchNormalization()(x); x=layers.UpSampling1D(2)(x)
    x=layers.Conv1D(16,7,activation='relu',padding='same')(x)
    x=layers.BatchNormalization()(x)
    out=layers.Conv1D(1,3,activation='linear',padding='same')(x)
    assert out.shape[1]==L
    m=models.Model(inp,out)
    m.compile(optimizer=tf.keras.optimizers.Adam(5e-4),loss='mse',metrics=['mae'])
    return m

K.clear_session(); gc.collect()
model=build_autoencoder(tl,latent=LATENT_DIM)
model.summary()

# ════════════════════════════════════════════════════════
#  STEP 7 — TRAIN
# ════════════════════════════════════════════════════════
section_title("STEP 7 — TRAIN")

bs=max(1,min(BATCH_SIZE,len(train_idx)))
mon='val_loss' if set(val_idx)!=set(train_idx) else 'loss'
history=model.fit(
    X_std3[train_idx],X_std3[train_idx],
    validation_data=(X_std3[val_idx],X_std3[val_idx]),
    epochs=EPOCHS,batch_size=bs,
    callbacks=[
        callbacks.EarlyStopping(monitor=mon,patience=20,
                                restore_best_weights=True,min_delta=1e-6,verbose=1),
        callbacks.ReduceLROnPlateau(monitor=mon,factor=0.5,
                                    patience=8,min_lr=1e-7,verbose=1),
        callbacks.LambdaCallback(
            on_epoch_end=lambda e,l: gc.collect() if e%10==0 else None),
    ],verbose=1)

# Training curves
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,4.5))
fig.suptitle('Training History — Ternary Spectra',fontsize=13,fontweight='bold')
er=range(len(history.history['loss']))
for ax,key,title in [(ax1,'loss','MSE Loss'),(ax2,'mae','MAE')]:
    ax.plot(er,history.history[key],color=TRAIN_COL,lw=2,label='Train')
    ax.plot(er,history.history[f'val_{key}'],color=VAL_COL,lw=2,ls='--',label='Val')
    ax.set_title(title,fontsize=11,fontweight='bold')
    ax.set_xlabel('Epoch',fontsize=10,fontweight='bold')
    leg=ax.legend(fontsize=9,loc='best')
    for t in leg.get_texts(): t.set_fontweight('bold')
    bold_ticks(ax)
ax1.set_ylabel('Loss',fontsize=10,fontweight='bold'); ax1.set_yscale('log')
ax2.set_ylabel('MAE',fontsize=10,fontweight='bold')
plt.tight_layout(pad=0.5,w_pad=1.0)
save_figure(fig,'fig2_training_curves')

# ════════════════════════════════════════════════════════
#  STEP 8 — RECONSTRUCT (clip-only)
# ════════════════════════════════════════════════════════
section_title("STEP 8 — RECONSTRUCT")

CHUNK=100; X_recon=np.zeros_like(X)
for start in range(0,len(X),CHUNK):
    end=min(start+CHUNK,len(X))
    raw=model.predict(X_std3[start:end],verbose=0)
    for j in range(end-start):
        gi=start+j
        inv=scalers[gi].inverse_transform(
                raw[j,:,0].reshape(-1,1)).ravel().astype(np.float32)
        X_recon[gi]=np.clip(inv,0.0,1.0)
    if end%500==0 or end==len(X):
        print(f"  Reconstructed {end}/{len(X)}...")
    gc.collect()
print(f"✓ Recon range [{X_recon.min():.6f}, {X_recon.max():.6f}]")

# Per-sample stats
per_r2,per_rmse=[],[]
for i in range(len(X)):
    yt,yp=X[i],X_recon[i]
    if np.std(yp)<1e-10 or np.std(yt)<1e-10: r2i=0.0
    else: ri,_=pearsonr(yt,yp); r2i=float(ri**2)
    per_r2.append(r2i); per_rmse.append(float(np.sqrt(np.mean((yt-yp)**2))))

r2_arr=np.array(per_r2)
pct25,pct50,pct75,pct90=(np.percentile(r2_arr,p) for p in [25,50,75,90])
n_good=int(np.sum(r2_arr>=0.95)); n_fair=int(np.sum((r2_arr>=0.85)&(r2_arr<0.95)))
n_poor=int(np.sum(r2_arr<0.85)); nt=len(r2_arr)
print(f"  R² Mean={r2_arr.mean():.4f}  Median={pct50:.4f}  "
      f"Min={r2_arr.min():.4f}  Max={r2_arr.max():.4f}")

def _quality(r2):
    return '✓ Good' if r2>=0.95 else ('~ Fair' if r2>=0.85 else '✗ Poor')
def _fill_alpha(r2):
    if r2>=0.95: return 0.14
    if r2>=0.85: return 0.09
    return 0.04

# ════════════════════════════════════════════════════════
#  DRAW HELPERS
#
#  Residual convention: residual is (true - pred), so under-
#  reconstructed absorbance peaks spike UPWARD, matching the
#  absorbance display convention, instead of dipping down
#  like transmittance.
#
#  Legend placement: every in-axes legend below uses
#  loc='best' instead of a hardcoded corner, so it
#  automatically avoids whichever part of each individual
#  spectrum's curve is tall, rather than a fixed corner that
#  works for some molecules and overlaps the data on others.
#
#  BOLD: every axis label, tick label, title, legend entry
#  and annotation box in the helpers below is explicitly set
#  to bold (on top of the global rcParams bold default).
# ════════════════════════════════════════════════════════
def draw_spectrum(ax,idx,label='',compact=False):
    yt=X[idx]; yp=X_recon[idx]
    col=VAL_COL if idx in val_idx else TRAIN_COL
    fs=8 if compact else 9.5
    r2i=per_r2[idx]; rmsei=per_rmse[idx]; q=_quality(r2i); fa=_fill_alpha(r2i)
    yt_d=_yd(yt); yp_d=_yd(yp)
    ax.fill_between(common_wn,yt_d,yp_d,color=col,alpha=fa,label='Deviation',zorder=1)
    ax.plot(common_wn,yp_d,color=col,lw=1.4,alpha=0.90,label='Reconstructed',zorder=3)
    ax.plot(common_wn,yt_d,color='#111111',lw=1.3,alpha=0.92,label='Original',zorder=4)
    ax.text(0.02,0.97,f'R²={r2i:.4f}  {q}\nRMSE={rmsei:.4f}',
            transform=ax.transAxes,fontsize=fs,va='top',fontweight='bold',
            bbox=dict(facecolor='white',edgecolor='#aaaaaa',boxstyle='round,pad=0.35',alpha=0.95))
    ax.set_xlim(common_wn.min(),common_wn.max()); ax.set_ylim(*_ylim()); ax.set_yticks(_yticks())
    ax.set_xlabel('Wavenumber (cm⁻¹)',fontsize=fs+0.5,fontweight='bold')
    ax.set_ylabel(_ylabel(),fontsize=fs+0.5,fontweight='bold')
    ax.tick_params(labelsize=fs-0.5)
    bold_ticks(ax)
    tc=GOOD_COL if r2i>=0.95 else (FAIR_COL if r2i>=0.85 else POOR_COL)
    ax.set_title(label if label else names[idx],fontsize=fs+0.5,fontweight='bold',color=tc,pad=4)
    leg=ax.legend(fontsize=fs-0.5,loc='best',handlelength=1.4,framealpha=0.92)
    for t in leg.get_texts(): t.set_fontweight('bold')

def draw_residual(ax,idx,label='',compact=False):
    yt=X[idx]; yp=X_recon[idx]; res=yt-yp    # true - pred
    rs=float(np.std(res)); rm=float(np.mean(res)); fs=8 if compact else 9.5
    ax.plot(common_wn,res,color=RES_COL,lw=1.1,alpha=0.88,label='Residual (true−pred)',zorder=3)
    ax.axhline(0,color=GOOD_COL,lw=1.5,ls='--',label='Zero (ideal)',zorder=2)
    ax.axhspan(-rs,rs,color=RES_COL,alpha=0.15,label='±1σ',zorder=1)
    ax.axhspan(-2*rs,2*rs,color=RES_COL,alpha=0.07,label='±2σ',zorder=0)
    win=max(1,len(res)//50); sm=np.convolve(res,np.ones(win)/win,mode='same')
    ax.plot(common_wn,sm,color=ACCENT,lw=1.6,ls='-.',alpha=0.85,label=f'Trend (w={win})',zorder=4)
    herr=np.abs(res)>2*rs
    if herr.any():
        ax.scatter(common_wn[herr],res[herr],color='#d62728',s=7,zorder=5,alpha=0.65,
                   label=f'|Δ|>2σ ({herr.sum()} pts)')
    pct2=100.0*float(np.mean(herr))
    ax.text(0.02,0.97,f'Mean={rm:+.4f}  Std={rs:.4f}\n% pts |Δ|>2σ : {pct2:.1f}%',
            transform=ax.transAxes,fontsize=fs,va='top',fontweight='bold',
            bbox=dict(facecolor='#fff8f0',edgecolor=RES_COL,boxstyle='round,pad=0.35',alpha=0.95))
    ax.set_xlim(common_wn.min(),common_wn.max())
    ax.set_xlabel('Wavenumber (cm⁻¹)',fontsize=fs+0.5,fontweight='bold')
    ax.set_ylabel('Residual',fontsize=fs+0.5,fontweight='bold')
    ax.tick_params(labelsize=fs-0.5)
    bold_ticks(ax)
    ax.set_title(label if label else 'Residual Spectrum',fontsize=fs+0.5,fontweight='bold',pad=4)
    leg=ax.legend(fontsize=fs-1,loc='best',handlelength=1.4,framealpha=0.92,ncol=2)
    for t in leg.get_texts(): t.set_fontweight('bold')

# ════════════════════════════════════════════════════════
#  FIGURES — Reconstructions grid
# ════════════════════════════════════════════════════════
section_title("FIGURES — SPECTRAL RECONSTRUCTIONS")

n_show=min(PREVIEW_N,len(X)); pidx=np.linspace(0,len(X)-1,n_show,dtype=int)
nc3=min(3,n_show); nr3=int(np.ceil(n_show/nc3))
fig=plt.figure(figsize=(18,3.4*nr3))
fig.suptitle('Spectral Reconstructions — Ternary (Absorbance [0,1])',fontsize=12,fontweight='bold',y=1.01)
for pi,di in enumerate(pidx):
    ax=fig.add_subplot(nr3,nc3,pi+1)
    col=VAL_COL if di in val_idx else TRAIN_COL; tag='VAL' if di in val_idx else 'TRAIN'
    r2i=per_r2[di]; fa=_fill_alpha(r2i)
    ax.fill_between(common_wn,_yd(X[di]),_yd(X_recon[di]),color=col,alpha=fa,zorder=1)
    ax.plot(common_wn,_yd(X_recon[di]),color=col,lw=1.0,alpha=0.90,zorder=3,label='Recon')
    ax.plot(common_wn,_yd(X[di]),color='#222222',lw=0.9,alpha=0.90,zorder=4,label='Orig')
    ax.set_xlim(common_wn.min(),common_wn.max()); ax.set_ylim(*_ylim())
    ax.set_yticks([0,0.5,1.0])
    xt=np.linspace(common_wn.min(),common_wn.max(),4); ax.set_xticks(xt)
    ax.set_xticklabels([f"{v:.0f}" for v in xt],fontsize=5.5,rotation=30,ha='right',fontweight='bold')
    ax.tick_params(labelsize=5.5)
    bold_ticks(ax)
    ax.set_xlabel('Wavenumber (cm⁻¹)',fontsize=6,labelpad=1,fontweight='bold')
    ax.set_ylabel(_ylabel(),fontsize=6,labelpad=1,fontweight='bold')
    tc=GOOD_COL if r2i>=0.95 else (FAIR_COL if r2i>=0.85 else POOR_COL)
    ax.set_title(f"{names[di][:22]}\n[{tag}] R²={r2i:.3f}",fontsize=6,color=tc,fontweight='bold')
    leg=ax.legend(fontsize=4.5,loc='best',handlelength=1)
    for t in leg.get_texts(): t.set_fontweight('bold')
for j in range(n_show,nr3*nc3): fig.add_subplot(nr3,nc3,j+1).set_visible(False)
plt.tight_layout(pad=0.3,h_pad=0.6,w_pad=0.4)
save_figure(fig,'fig3_spectral_reconstructions')

# ── Regression hexbin ────────────────────────────────
section_title("FIGURES — REGRESSION")

MAX_PTS=300_000; n_total=len(X)*tl; step=max(1,n_total//MAX_PTS)
yt_all=X[:,::step].ravel(); yp_all=X_recon[:,::step].ravel()
yt_tr=X[train_idx,::step].ravel(); yp_tr=X_recon[train_idx,::step].ravel()
yt_vl=X[val_idx,::step].ravel();   yp_vl=X_recon[val_idx,::step].ravel()

r_g,_=pearsonr(yt_all,yp_all); r2_all=float(r_g**2)
rmse_gl=float(np.sqrt(np.mean((yt_all-yp_all)**2)))
rc=np.polyfit(yt_all,yp_all,1)
res_gl=yt_all-yp_all                    # true - pred
rsd_gl=float(np.std(res_gl))
rc_res=np.polyfit(yt_all,res_gl,1)
pad=0.03; dmin=min(yt_all.min(),yp_all.min())-pad; dmax=max(yt_all.max(),yp_all.max())+pad
xl=np.linspace(dmin,dmax,300); fv=np.poly1d(rc)(xl)

fig,axes=plt.subplots(1,2,figsize=(13,5.5),sharey=True)
fig.suptitle(f'Regression  ·  R²={r2_all:.4f}  ·  RMSE={rmse_gl:.4f}  ·  {len(X)} spectra',
             fontsize=12,fontweight='bold')
for ax,(yt_s,yp_s,col,cmap),title in zip(
        axes,
        [(yt_tr,yp_tr,TRAIN_COL,'Blues'),(yt_vl,yp_vl,VAL_COL,'Reds')],
        [f'Train  (n={len(train_idx)})',f'Val    (n={len(val_idx)})']):
    hb=ax.hexbin(yt_s,yp_s,gridsize=60,cmap=cmap,mincnt=1,linewidths=0.2)
    cb=fig.colorbar(hb,ax=ax,pad=0.02,shrink=0.85)
    cb.set_label('Point density',fontsize=8,fontweight='bold')
    cb.ax.tick_params(labelsize=7)
    for t in cb.ax.get_yticklabels(): t.set_fontweight('bold')
    ax.plot(xl,xl,color=GOOD_COL,lw=1.8,ls='--',label='Ideal y=x')
    ax.plot(xl,fv,color='#333333',lw=1.6,label=f'Fit y={rc[0]:.3f}x{rc[1]:+.3f}')
    ax.fill_between(xl,fv-rsd_gl,fv+rsd_gl,alpha=0.10,color='#333333',label='±1σ')
    if len(yt_s)>5:
        rs,_=pearsonr(yt_s,yp_s); r2s=float(rs**2); rms=float(np.sqrt(np.mean((yt_s-yp_s)**2)))
    else: r2s,rms=r2_all,rmse_gl
    ax.text(0.03,0.97,f'R²={r2s:.4f}\nRMSE={rms:.4f}',transform=ax.transAxes,fontsize=9,va='top',
            fontweight='bold',
            bbox=dict(facecolor='white',edgecolor='#aaaaaa',boxstyle='round,pad=0.4',alpha=0.95))
    ax.set_xlim(dmin,dmax); ax.set_ylim(dmin,dmax)
    ax.set_xlabel('Original Absorbance',fontsize=10,fontweight='bold')
    ax.set_title(title,fontsize=11,fontweight='bold',color=col)
    leg=ax.legend(fontsize=8.5,loc='best',handlelength=1.5)
    for t in leg.get_texts(): t.set_fontweight('bold')
    ax.tick_params(labelsize=9)
    bold_ticks(ax)
axes[0].set_ylabel('Reconstructed Absorbance',fontsize=10,fontweight='bold')
plt.tight_layout(pad=0.5,w_pad=0.8)
save_figure(fig,'fig4_regression_hexbin')

# ── Residual scatter ─────────────────────────────────
fig,ax=plt.subplots(figsize=(9,4.5))
stride2=max(1,len(X)//80)
for i in range(0,len(X),stride2):
    c=TRAIN_COL if i in train_idx else VAL_COL
    ax.scatter(X[i,::step],X[i,::step]-X_recon[i,::step],alpha=0.06,s=1.0,color=c,rasterized=True)  # true - pred
ax.axhline(0,color=GOOD_COL,lw=1.8,ls='--',label='Zero (ideal)')
for ns,al,lb in [(1,0.14,'±1σ'),(2,0.06,'±2σ')]:
    ax.axhspan(-ns*rsd_gl,ns*rsd_gl,color=RES_COL,alpha=al,label=lb)
ax.plot(xl,np.poly1d(rc_res)(xl),color=RES_COL,lw=2,label=f'Trend={rc_res[0]:.4f}')
leg_h=[Line2D([0],[0],marker='o',color='w',markerfacecolor=TRAIN_COL,markersize=8,label=f'Train ({len(train_idx)})'),
     Line2D([0],[0],marker='o',color='w',markerfacecolor=VAL_COL,markersize=8,label=f'Val ({len(val_idx)})')]
h,_=ax.get_legend_handles_labels()
leg=ax.legend(handles=leg_h+h,fontsize=9,loc='best')
for t in leg.get_texts(): t.set_fontweight('bold')
ax.set_xlabel('Original Absorbance',fontsize=10,fontweight='bold')
ax.set_ylabel('Residual',fontsize=10,fontweight='bold')
ax.set_title('Residual Analysis',fontsize=11,fontweight='bold')
bold_ticks(ax)
plt.tight_layout(pad=0.5); save_figure(fig,'fig5_residual_scatter')

# ── Per-sample R² bar ────────────────────────────────
BAR_CAP=min(80,len(X)); so=np.argsort(r2_arr)[::-1]
s_r2=r2_arr[so[:BAR_CAP]]; s_names=[names[i] for i in so[:BAR_CAP]]
fig,ax=plt.subplots(figsize=(max(13,BAR_CAP*0.35),5.5))
bc=[GOOD_COL if v>=0.95 else FAIR_COL if v>=0.85 else POOR_COL for v in s_r2]
bars=ax.bar(range(len(s_r2)),s_r2,color=bc,edgecolor='white',width=0.72)
for pv,pl,pc in [(pct90,'90th','#7b2d8b'),(pct75,'75th','#1f77b4'),
                 (pct50,'Med','#e07b00'),(pct25,'25th','#d62728')]:
    ax.axhline(pv,color=pc,lw=1.1,ls=':',alpha=0.9)
    ax.text(len(s_r2)-0.4,pv+0.007,f'{pl}: {pv:.3f}',color=pc,fontsize=7.5,ha='right',fontweight='bold')
for bar,val,idx in zip(bars,s_r2,so[:BAR_CAP]):
    tc=TRAIN_COL if idx in train_idx else VAL_COL
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.007,
            f'{val:.3f}',ha='center',va='bottom',fontsize=7,fontweight='bold')
    ax.text(bar.get_x()+bar.get_width()/2,0.01,
            'V' if idx in val_idx else 'T',ha='center',va='bottom',
            fontsize=6.5,color=tc,fontweight='bold')
ax.set_xticks(range(len(s_names)))
ax.set_xticklabels([n[:20] for n in s_names],rotation=45,ha='right',fontsize=7,fontweight='bold')
ax.set_ylim(0,1.12); ax.set_ylabel('R²',fontsize=10,fontweight='bold')
ax.set_title(f'Per-Sample R²  ·  {len(X)} Spectra',fontsize=11,fontweight='bold')
ax.axhline(0.95,color=GOOD_COL,lw=1.1,ls='--',alpha=0.6)
ax.axhline(0.85,color=FAIR_COL,lw=1.1,ls='--',alpha=0.6)
leg=ax.legend(handles=[Patch(facecolor=GOOD_COL,label='Good ≥0.95'),
                   Patch(facecolor=FAIR_COL,label='Fair 0.85–0.95'),
                   Patch(facecolor=POOR_COL,label='Poor <0.85')],fontsize=8.5,loc='best')
for t in leg.get_texts(): t.set_fontweight('bold')
bold_ticks(ax)
plt.tight_layout(pad=0.5); save_figure(fig,'fig6_r2_bar')

# ── R² distribution ──────────────────────────────────
fig,ax=plt.subplots(figsize=(8,4))
bins=min(60,max(10,len(X)//20))
counts,_,_=ax.hist(r2_arr,bins=bins,color=ACCENT,edgecolor='white',alpha=0.75)
cm=float(max(counts)) if len(counts)>0 and max(counts)>0 else 1.0
for pv,pl,pc in [(pct25,'25th','#d62728'),(pct50,'Med','#e07b00'),
                 (pct75,'75th','#1f77b4'),(pct90,'90th','#7b2d8b')]:
    ax.axvline(pv,color=pc,lw=2.5,ls='--')
    ax.text(pv,cm*0.92,f'{pl}\n{pv:.3f}',color=pc,fontsize=8,va='top',ha='left',fontweight='bold')
ax.axvspan(0.95,1.01,alpha=0.07,color=GOOD_COL,label='Good ≥0.95')
ax.axvspan(0.85,0.95,alpha=0.07,color=FAIR_COL,label='Fair ≥0.85')
ax.axvspan(0.0,0.85,alpha=0.07,color=POOR_COL,label='Poor <0.85')
ax.text(0.02,0.97,
        f"  N={nt}\n  Good: {n_good} ({100*n_good//max(nt,1)}%)\n"
        f"  Fair: {n_fair} ({100*n_fair//max(nt,1)}%)\n"
        f"  Poor: {n_poor} ({100*n_poor//max(nt,1)}%)\n\n"
        f"  Mean={r2_arr.mean():.4f}  Std={r2_arr.std():.4f}",
        transform=ax.transAxes,fontsize=9,va='top',fontweight='bold',
        bbox=dict(facecolor='#f5f5f5',edgecolor='#cccccc',boxstyle='round,pad=0.5',alpha=0.95))
leg=ax.legend(fontsize=9,loc='best')
for t in leg.get_texts(): t.set_fontweight('bold')
ax.set_xlabel('R² per Spectrum',fontsize=10,fontweight='bold')
ax.set_ylabel('Count',fontsize=10,fontweight='bold')
ax.set_xlim(-0.02,1.02); ax.set_title('R² Distribution — Ternary',fontsize=11,fontweight='bold')
bold_ticks(ax)
plt.tight_layout(pad=0.5); save_figure(fig,'fig7_r2_distribution')

# ════════════════════════════════════════════════════════
#  FIG 8 — 2×2  Best Good-1 + Good-2
# ════════════════════════════════════════════════════════
STD_MIN          = 0.08
PEAK_MIN         = 3
PEAK_HEIGHT_THRESH = 0.10
PEAK_SPREAD_THRESH = 400.0

EXCLUDE_GOOD_KW = [
    'dimethyl_sulphide','dimethyl_sulfide','dimethylsulphide',
    'dimethylsulfide','dms_','_dms','dimethylsulph','dimethylsulf',
    '_n2_','_n2.','n2_','_ar_','_ar.','_he_','_kr_','_xe_',
    '_co_gas','co_gas_',
]

def _excluded_good(i):
    n = names[i].lower()
    return any(kw in n for kw in EXCLUDE_GOOD_KW)

def _spectral_quality_score(i):
    s    = X[i]
    std  = float(np.std(s))

    if is_ramp_spectrum(s):
        return False, 0, std, 0.0

    if std < STD_MIN:
        return False, 0, std, 0.0

    win    = max(5, len(s)//80)
    peaks  = []
    for j in range(win, len(s)-win):
        seg = s[j-win:j+win+1]
        if s[j] == seg.max() and (s[j] - seg.min()) > PEAK_HEIGHT_THRESH:
            peaks.append(j)

    n_peaks = len(peaks)
    if n_peaks < PEAK_MIN:
        return False, n_peaks, std, 0.0

    peak_wns = common_wn[peaks]
    spread   = float(peak_wns.max() - peak_wns.min())
    passes   = spread >= PEAK_SPREAD_THRESH
    return passes, n_peaks, std, spread

all_i  = list(range(len(X)))
med_r2 = np.median(r2_arr)

print("\n  ── Good-spotlight quality screening ────────────────")
quality_results = {}
for i in all_i:
    passes, np_, std, spread = _spectral_quality_score(i)
    quality_results[i] = {
        'passes':  passes,
        'n_peaks': np_,
        'std':     std,
        'spread':  spread,
    }

n_pass = sum(1 for v in quality_results.values() if v['passes'])
print(f"  {n_pass}/{len(all_i)} spectra pass spectral quality filter "
      f"(std>{STD_MIN}, peaks≥{PEAK_MIN}, spread>{PEAK_SPREAD_THRESH} cm⁻¹)")

desc_all = sorted(all_i, key=lambda i: r2_arr[i], reverse=True)
asc_all  = sorted(all_i, key=lambda i: r2_arr[i])
med_all  = sorted(all_i, key=lambda i: abs(r2_arr[i]-med_r2))

good_candidates = [
    i for i in desc_all
    if not _excluded_good(i) and quality_results[i]['passes']
]

if len(good_candidates) >= 2:
    good2 = good_candidates[:2]
    print(f"  Good picks: both from quality-filtered list ({len(good_candidates)} eligible)")
elif len(good_candidates) == 1:
    fallback = [i for i in desc_all if not _excluded_good(i) and i not in good_candidates]
    good2 = good_candidates + fallback[:1]
    print(f"  Good picks: 1 quality-filtered + 1 fallback (shape filter relaxed)")
else:
    good2 = desc_all[:2]
    print(f"  Good picks: top-2 by R² (all filters relaxed — too few candidates)")

for i in good2:
    qr = quality_results[i]
    print(f"    {names[i][:45]}")
    print(f"      R²={r2_arr[i]:.4f}  std={qr['std']:.3f}  "
          f"peaks={qr['n_peaks']}  spread={qr['spread']:.0f} cm⁻¹")

poor2 = asc_all[:2]
mid2  = [i for i in med_all if i not in good2 and i not in poor2][:2]
if len(mid2) < 2: mid2 = med_all[:2]

section_title("FIG 10 — 6 INDIVIDUAL SPOTLIGHTS")

OH_MIN,OH_MAX=2500.0,3500.0; FP_MIN,FP_MAX=800.0,1500.0
FP_BANDS={'C–O':1050,'C–C':1150,'O–H b':1280,'C–H b':1450}

def plot_4panel_spotlight(idx, fig_suffix):
    yt=X[idx]; yp=X_recon[idx]; res=yt-yp    # true - pred
    rs=float(np.std(res)); rm=float(np.mean(res))
    r2m=per_r2[idx]; rmse_m=per_rmse[idx]
    q=_quality(r2m); fa=_fill_alpha(r2m)
    spl='TRAIN' if idx in train_idx else 'VAL'
    col=TRAIN_COL if spl=='TRAIN' else VAL_COL
    ul=conversion_log[idx]['unit'] if idx<len(conversion_log) else 'Absorbance'
    tc=GOOD_COL if r2m>=0.95 else (FAIR_COL if r2m>=0.85 else POOR_COL)

    def reg_r2(lo,hi):
        m=(common_wn>=lo)&(common_wn<=hi)
        if m.sum()<5: return float('nan')
        ytr,ypr=yt[m],yp[m]
        if np.std(ypr)<1e-10: return 0.0
        r,_=pearsonr(ytr,ypr); return float(r**2)

    fig=plt.figure(figsize=(14,9))
    fig.suptitle(f'Spotlight  ·  [{spl}]  {names[idx][:50]}\n'
                 f'R²={r2m:.4f}    RMSE={rmse_m:.4f}    {q}  |  {ul}',
                 fontsize=11,fontweight='bold',y=1.01,color=tc)
    gs=gridspec.GridSpec(2,2,figure=fig,hspace=0.34,wspace=0.26,
                         left=0.08,right=0.97,top=0.91,bottom=0.09)
    # (a) Full spectrum
    ax_a=fig.add_subplot(gs[0,0])
    ax_a.axvspan(OH_MIN,min(OH_MAX,common_wn.max()),color='#d62728',alpha=0.06,zorder=0)
    ax_a.axvspan(max(FP_MIN,common_wn.min()),FP_MAX,color='#1f77b4',alpha=0.06,zorder=0)
    ax_a.fill_between(common_wn,_yd(yt),_yd(yp),color=col,alpha=fa,label='Deviation',zorder=1)
    ax_a.plot(common_wn,_yd(yp),color=col,lw=1.4,alpha=0.88,label='Reconstructed',zorder=3)
    ax_a.plot(common_wn,_yd(yt),color='#111111',lw=1.3,alpha=0.93,label='Original',zorder=4)
    ax_a.set_xlim(common_wn.min(),common_wn.max()); ax_a.set_ylim(*_ylim())
    ax_a.set_xlabel('Wavenumber (cm⁻¹)',fontsize=10,fontweight='bold')
    ax_a.set_ylabel(_ylabel(),fontsize=10,fontweight='bold')
    ax_a.set_title(f'(a) Full Spectrum — {names[idx][:35]} [{spl}]',
                   fontsize=10,fontweight='bold',color=tc,pad=4)
    ax_a.text(0.02,0.97,f'R²={r2m:.4f}  {q}\nRMSE={rmse_m:.4f}',
              transform=ax_a.transAxes,fontsize=8.5,va='top',fontweight='bold',
              bbox=dict(facecolor='white',edgecolor='#aaaaaa',boxstyle='round,pad=0.35',alpha=0.95))
    if r2m<0.85:
        ax_a.text(0.50,0.97,'⚠ Poor reconstruction\nOriginal shown in black\nReconstructed in colour',
                  transform=ax_a.transAxes,fontsize=8,va='top',ha='center',
                  color=POOR_COL,fontweight='bold',
                  bbox=dict(facecolor='#fff0f0',edgecolor=POOR_COL,boxstyle='round,pad=0.35',alpha=0.95))
    leg_h=[Line2D([0],[0],color='#111111',lw=1.3,label='Original'),
           Line2D([0],[0],color=col,lw=1.4,label='Reconstructed'),
           Patch(facecolor=col,alpha=fa*3,label='Deviation'),
           Patch(facecolor='#d62728',alpha=0.25,label='O–H region'),
           Patch(facecolor='#1f77b4',alpha=0.25,label='C–H/C–O')]
    leg=ax_a.legend(handles=leg_h,fontsize=7.5,loc='best',handlelength=1.3,ncol=2)
    for t in leg.get_texts(): t.set_fontweight('bold')
    ax_a.tick_params(labelsize=9); bold_ticks(ax_a)
    # (b) Residual
    ax_b=fig.add_subplot(gs[1,0])
    ax_b.plot(common_wn,res,color=RES_COL,lw=1.1,alpha=0.88,label='Residual (true−pred)')
    ax_b.axhline(0,color=GOOD_COL,lw=1.5,ls='--',label='Zero (ideal)')
    ax_b.axhspan(-rs,rs,color=RES_COL,alpha=0.14,label='±1σ')
    ax_b.axhspan(-2*rs,2*rs,color=RES_COL,alpha=0.07,label='±2σ')
    win=max(1,len(res)//50); sm=np.convolve(res,np.ones(win)/win,mode='same')
    ax_b.plot(common_wn,sm,color=ACCENT,lw=1.6,ls='-.',alpha=0.85,label=f'Trend (w={win})')
    herr=np.abs(res)>2*rs
    if herr.any():
        ax_b.scatter(common_wn[herr],res[herr],color='#d62728',s=7,zorder=5,alpha=0.65,
                     label=f'|Δ|>2σ ({herr.sum()} pts)')
    ax_b.set_xlim(common_wn.min(),common_wn.max())
    ax_b.set_xlabel('Wavenumber (cm⁻¹)',fontsize=10,fontweight='bold')
    ax_b.set_ylabel('Residual',fontsize=10,fontweight='bold')
    ax_b.set_title('(b) Residual Spectrum',fontsize=10,fontweight='bold',pad=4)
    pct2=100*float(np.mean(herr))
    ax_b.text(0.02,0.97,f'Mean={rm:+.4f}  Std={rs:.4f}\n% pts |Δ|>2σ: {pct2:.1f}%',
              transform=ax_b.transAxes,fontsize=8.5,va='top',fontweight='bold',
              bbox=dict(facecolor='#fff8f0',edgecolor=RES_COL,boxstyle='round,pad=0.35',alpha=0.95))
    leg=ax_b.legend(fontsize=7.5,loc='best',handlelength=1.3,ncol=2)
    for t in leg.get_texts(): t.set_fontweight('bold')
    ax_b.tick_params(labelsize=9); bold_ticks(ax_b)
    # (c) O-H zoom
    ax_c=fig.add_subplot(gs[0,1])
    m_oh=(common_wn>=OH_MIN)&(common_wn<=OH_MAX); wn_oh=common_wn[m_oh]
    if m_oh.sum()>5:
        ax_c.fill_between(wn_oh,_yd(yt[m_oh]),_yd(yp[m_oh]),color=col,alpha=fa,label='Deviation')
        ax_c.plot(wn_oh,_yd(yp[m_oh]),color=col,lw=1.4,alpha=0.88,label='Reconstructed')
        ax_c.plot(wn_oh,_yd(yt[m_oh]),color='#111111',lw=1.3,alpha=0.93,label='Original')
        ax_c.set_xlim(wn_oh.min(),wn_oh.max())
        r2oh=reg_r2(OH_MIN,OH_MAX)
        ax_c.text(0.03,0.97,f'O–H region  R²={r2oh:.4f}' if not np.isnan(r2oh) else 'N/A',
                  transform=ax_c.transAxes,fontsize=8.5,va='top',color='#d62728',fontweight='bold',
                  bbox=dict(facecolor='#fff0f0',edgecolor='#d62728',boxstyle='round,pad=0.3',alpha=0.9))
    else:
        ax_c.text(0.5,0.5,'Not in range',ha='center',va='center',transform=ax_c.transAxes,fontsize=10,fontweight='bold')
    ax_c.set_ylim(*_ylim())
    ax_c.set_xlabel('Wavenumber (cm⁻¹)',fontsize=10,fontweight='bold')
    ax_c.set_ylabel(_ylabel(),fontsize=10,fontweight='bold')
    ax_c.set_title(f'(c) O–H Stretch Zoom ({OH_MIN:.0f}–{OH_MAX:.0f} cm⁻¹)',
                   fontsize=10,fontweight='bold',color='#d62728',pad=4)
    leg=ax_c.legend(fontsize=8,loc='best',handlelength=1.3)
    for t in leg.get_texts(): t.set_fontweight('bold')
    ax_c.tick_params(labelsize=9); bold_ticks(ax_c)
    # (d) Fingerprint zoom
    ax_d=fig.add_subplot(gs[1,1])
    m_fp=(common_wn>=FP_MIN)&(common_wn<=FP_MAX); wn_fp=common_wn[m_fp]
    if m_fp.sum()>5:
        ax_d.fill_between(wn_fp,_yd(yt[m_fp]),_yd(yp[m_fp]),color=col,alpha=fa,label='Deviation')
        ax_d.plot(wn_fp,_yd(yp[m_fp]),color=col,lw=1.4,alpha=0.88,label='Reconstructed')
        ax_d.plot(wn_fp,_yd(yt[m_fp]),color='#111111',lw=1.3,alpha=0.93,label='Original')
        ax_d.set_xlim(wn_fp.min(),wn_fp.max())
        for bname,bwn in FP_BANDS.items():
            if FP_MIN<=bwn<=FP_MAX:
                ax_d.axvline(bwn,color='#888888',lw=0.8,ls='--',alpha=0.55)
                ax_d.text(bwn,1.06,bname,ha='center',va='bottom',fontsize=6,
                          color='#555555',rotation=90,fontweight='bold',
                          transform=ax_d.get_xaxis_transform())
        r2fp=reg_r2(FP_MIN,FP_MAX)
        ax_d.text(0.03,0.97,f'C–H/C–O region  R²={r2fp:.4f}' if not np.isnan(r2fp) else 'N/A',
                  transform=ax_d.transAxes,fontsize=8.5,va='top',color='#1f77b4',fontweight='bold',
                  bbox=dict(facecolor='#f0f4ff',edgecolor='#1f77b4',boxstyle='round,pad=0.3',alpha=0.9))
    else:
        ax_d.text(0.5,0.5,'Not in range',ha='center',va='center',transform=ax_d.transAxes,fontsize=10,fontweight='bold')
    ax_d.set_ylim(*_ylim())
    ax_d.set_xlabel('Wavenumber (cm⁻¹)',fontsize=10,fontweight='bold')
    ax_d.set_ylabel(_ylabel(),fontsize=10,fontweight='bold')
    ax_d.set_title(f'(d) C–H/C–O Fingerprint ({FP_MIN:.0f}–{FP_MAX:.0f} cm⁻¹)',
                   fontsize=10,fontweight='bold',color='#1f77b4',pad=4)
    leg=ax_d.legend(fontsize=8,loc='best',handlelength=1.3)
    for t in leg.get_texts(): t.set_fontweight('bold')
    ax_d.tick_params(labelsize=9); bold_ticks(ax_d)
    save_figure(fig,f'fig10_{fig_suffix}')

spotlight_6=[(good2[0],'spotlight_good1'),(good2[1],'spotlight_good2'),
             (mid2[0],'spotlight_mid1'),  (mid2[1],'spotlight_mid2'),
             (poor2[0],'spotlight_poor1'),(poor2[1],'spotlight_poor2')]

print(f"\n  6 spotlights:")
for idx,suf in spotlight_6:
    spl='VAL' if idx in val_idx else 'TRAIN'
    print(f"    {suf:<22}: R²={r2_arr[idx]:.4f}  [{spl}]  {names[idx][:40]}")

for idx,suf in spotlight_6:
    plot_4panel_spotlight(idx,suf)

section_title("FIG 8 — 2×2 GOOD-1 / GOOD-2")

best_tr = good2[0]
best_vl = good2[1]
print(f"  Fig8 GOOD-1 → {names[best_tr][:50]}  R²={per_r2[best_tr]:.4f}")
print(f"  Fig8 GOOD-2 → {names[best_vl][:50]}  R²={per_r2[best_vl]:.4f}")

fig8=plt.figure(figsize=(15,10))
fig8.suptitle(
    f'Fig 8  ·  Representative Ternary Spectra  ·  Absorbance [0,1]\n'
    f'(a,b) Good Spectrum 1: {names[best_tr][:40]}    (c,d) Good Spectrum 2: {names[best_vl][:40]}',
    fontsize=10,fontweight='bold',y=1.01)
gs8=gridspec.GridSpec(2,2,figure=fig8,hspace=0.36,wspace=0.26,
                      left=0.08,right=0.97,top=0.91,bottom=0.08)
ax8a=fig8.add_subplot(gs8[0,0]); ax8b=fig8.add_subplot(gs8[1,0])
ax8c=fig8.add_subplot(gs8[0,1]); ax8d=fig8.add_subplot(gs8[1,1])
draw_spectrum(ax8a,best_tr,label=f'(a) Good 1 — {names[best_tr][:35]}\nOriginal vs Reconstructed')
draw_residual(ax8b,best_tr,label='(b) Good 1 — Residual Spectrum')
draw_spectrum(ax8c,best_vl,label=f'(c) Good 2 — {names[best_vl][:35]}\nOriginal vs Reconstructed')
draw_residual(ax8d,best_vl,label='(d) Good 2 — Residual Spectrum')
for ax,lbl in [(ax8a,'(a)'),(ax8b,'(b)'),(ax8c,'(c)'),(ax8d,'(d)')]:
    ax.text(-0.09,1.04,lbl,transform=ax.transAxes,fontsize=13,fontweight='bold',color='#111111')
save_figure(fig8,'fig8_good1_good2_2x2')

# ════════════════════════════════════════════════════════
#  FIG 9 — 3×2 SPOTLIGHT  Best / Moderate / Poor
# ════════════════════════════════════════════════════════
section_title("FIG 9 — 3×2 SPOTLIGHT")

all_i=list(range(len(X))); med_r2=np.median(r2_arr)

best_i = good2[0]
print(f"  Fig9 BEST → {names[best_i][:50]}  R²={r2_arr[best_i]:.4f}")

poor_i=min(all_i,key=lambda i: r2_arr[i])
mod_i=min(all_i,key=lambda i: abs(r2_arr[i]-med_r2))
if mod_i in (best_i,poor_i):
    cands=[i for i in all_i if i not in (best_i,poor_i)]
    mod_i=min(cands,key=lambda i: abs(r2_arr[i]-med_r2)) if cands else best_i

trio=[best_i,mod_i,poor_i]; trio_labels=['Best','Moderate','Poor']
trio_cols=[GOOD_COL,FAIR_COL,POOR_COL]
print(f"  Best    : {names[best_i][:45]}  R²={r2_arr[best_i]:.4f}")
print(f"  Moderate: {names[mod_i][:45]}   R²={r2_arr[mod_i]:.4f}")
print(f"  Poor    : {names[poor_i][:45]}  R²={r2_arr[poor_i]:.4f}")

fig9=plt.figure(figsize=(21,11))
fig9.suptitle('Fig 9  ·  Ternary Spotlight — Best / Moderate / Poor',
              fontsize=12,fontweight='bold',y=1.01)
gs9=gridspec.GridSpec(2,3,figure=fig9,hspace=0.36,wspace=0.26,
                      left=0.06,right=0.98,top=0.91,bottom=0.08)
for cj,(idx,lbl,col) in enumerate(zip(trio,trio_labels,trio_cols)):
    r2i=per_r2[idx]; q=_quality(r2i)
    ax_s=fig9.add_subplot(gs9[0,cj]); draw_spectrum(ax_s,idx,compact=True)
    ax_s.set_title(f'{lbl}  [{q}]\n{names[idx][:35]}\nR²={r2i:.4f}',
                   fontsize=8.5,fontweight='bold',color=col,pad=4)
    ax_r=fig9.add_subplot(gs9[1,cj])
    draw_residual(ax_r,idx,compact=True,label=f'Residual — {lbl}')
save_figure(fig9,'fig9_spotlight_bmp')

# ════════════════════════════════════════════════════════
#  FIG COMBINED — 4-PANEL SUMMARY  (a)(b)(c)(d)
#  FONT-SIZE FIX: this combined figure is a big (15x11 in)
#  2x2 grid, so its earlier tick/label/legend/annotation
#  sizes (8–10 pt) read far too small next to the other
#  figures in this script. Every text element in this
#  section is bumped up to match the readability of the
#  other multi-panel figures (fig8/fig9/fig10).
# ════════════════════════════════════════════════════════
section_title("FIG COMBINED — 4-PANEL SUMMARY (a-d)")

spot_idx = best_tr
spot_col = VAL_COL if spot_idx in val_idx else TRAIN_COL
spot_r2  = per_r2[spot_idx]; spot_rmse = per_rmse[spot_idx]
spot_q   = _quality(spot_r2); spot_fa = _fill_alpha(spot_r2)

figc = plt.figure(figsize=(15, 11))
gs_outer = gridspec.GridSpec(
    2, 2, figure=figc, hspace=0.42, wspace=0.28,
    left=0.07, right=0.97, top=0.93, bottom=0.07
)

# ── (a) top-left: best mixture's full spectrum ───────
yt_a = X[spot_idx]; yp_a = X_recon[spot_idx]
axc_a = figc.add_subplot(gs_outer[0, 0])
axc_a.axvspan(OH_MIN, min(OH_MAX, common_wn.max()), color='#d62728', alpha=0.06, zorder=0)
axc_a.axvspan(max(FP_MIN, common_wn.min()), FP_MAX,  color='#1f77b4', alpha=0.06, zorder=0)
axc_a.fill_between(common_wn, _yd(yt_a), _yd(yp_a), color=spot_col, alpha=spot_fa, zorder=1)
axc_a.plot(common_wn, _yd(yp_a), color=spot_col, lw=1.4, alpha=0.90, zorder=3)
axc_a.plot(common_wn, _yd(yt_a), color='#111111', lw=1.3, alpha=0.93, zorder=4)
axc_a.set_xlim(common_wn.min(), common_wn.max())
axc_a.set_ylim(*_ylim()); axc_a.set_yticks(_yticks())
axc_a.set_xlabel('Wavenumber (cm⁻¹)', fontsize=13, fontweight='bold')
axc_a.set_ylabel(_ylabel(), fontsize=13, fontweight='bold')
axc_a.tick_params(labelsize=12); bold_ticks(axc_a)
axc_a.text(0.02, 0.97, f'R²={spot_r2:.4f}  {spot_q}\nRMSE={spot_rmse:.4f}\n{names[spot_idx][:32]}',
           transform=axc_a.transAxes, fontsize=10.5, va='top', fontweight='bold',
           bbox=dict(facecolor='white', edgecolor='#aaaaaa', boxstyle='round,pad=0.35', alpha=0.95))
leg_a = [Line2D([0], [0], color='#111111', lw=1.3, label='Original'),
         Line2D([0], [0], color=spot_col, lw=1.4, label='Reconstructed'),
         Patch(facecolor=spot_col, alpha=spot_fa*3, label='Deviation'),
         Patch(facecolor='#d62728', alpha=0.25, label='O–H region'),
         Patch(facecolor='#1f77b4', alpha=0.25, label='C–H/C–O')]
leg=axc_a.legend(handles=leg_a, fontsize=9.5, loc='best', handlelength=1.3, framealpha=0.92)
for t in leg.get_texts(): t.set_fontweight('bold')
axc_a.text(-0.10, 1.05, '(a)', transform=axc_a.transAxes,
           fontsize=15, fontweight='bold', color='#111111')

# ── (b) top-right: that mixture's residual spectrum ──
res_b = yt_a - yp_a                     # true - pred
rs_b  = float(np.std(res_b)); rm_b = float(np.mean(res_b))
axc_b = figc.add_subplot(gs_outer[0, 1])
axc_b.plot(common_wn, res_b, color=RES_COL, lw=1.1, alpha=0.88, label='Residual', zorder=3)
axc_b.axhline(0, color=GOOD_COL, lw=1.5, ls='--', label='Zero (ideal)', zorder=2)
axc_b.axhspan(-rs_b, rs_b, color=RES_COL, alpha=0.15, label='±1σ', zorder=1)
axc_b.axhspan(-2*rs_b, 2*rs_b, color=RES_COL, alpha=0.07, label='±2σ', zorder=0)
win_b = max(1, len(res_b)//50)
sm_b  = np.convolve(res_b, np.ones(win_b)/win_b, mode='same')
axc_b.plot(common_wn, sm_b, color=ACCENT, lw=1.6, ls='-.', alpha=0.85, label=f'Trend (w={win_b})', zorder=4)
herr_b = np.abs(res_b) > 2*rs_b
if herr_b.any():
    axc_b.scatter(common_wn[herr_b], res_b[herr_b], color='#d62728', s=7,
                  zorder=5, alpha=0.65, label=f'|Δ|>2σ ({herr_b.sum()} pts)')
pct2_b = 100.0 * float(np.mean(herr_b))
axc_b.set_xlim(common_wn.min(), common_wn.max())
axc_b.set_xlabel('Wavenumber (cm⁻¹)', fontsize=13, fontweight='bold')
axc_b.set_ylabel('Residual', fontsize=13, fontweight='bold')
axc_b.tick_params(labelsize=12); bold_ticks(axc_b)
axc_b.text(0.02, 0.97, f'Mean={rm_b:+.4f}  Std={rs_b:.4f}\n% pts |Δ|>2σ: {pct2_b:.1f}%',
           transform=axc_b.transAxes, fontsize=10.5, va='top', fontweight='bold',
           bbox=dict(facecolor='#fff8f0', edgecolor=RES_COL, boxstyle='round,pad=0.35', alpha=0.95))
leg=axc_b.legend(fontsize=9.5, loc='best', handlelength=1.3, ncol=2, framealpha=0.92)
for t in leg.get_texts(): t.set_fontweight('bold')
axc_b.text(-0.08, 1.05, '(b)', transform=axc_b.transAxes,
           fontsize=15, fontweight='bold', color='#111111')

# ── (c) bottom-left: Train + Val regression hexbins ──
gs_c = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs_outer[1, 0], wspace=0.5)
axc_c1 = figc.add_subplot(gs_c[0])
axc_c2 = figc.add_subplot(gs_c[1])

t_sum = figc.text(0.29, 0.98, f'Regression  ·  R²={r2_all:.4f}  ·  RMSE={rmse_gl:.4f}  ·  {len(X)} spectra',
          ha='center', fontsize=12, fontweight='bold')

for ax, (yt_s, yp_s, cmap), title, title_col in zip(
        [axc_c1, axc_c2],
        [(yt_tr, yp_tr, 'Blues'), (yt_vl, yp_vl, 'Reds')],
        [f'Train  (n={len(train_idx)})', f'Val  (n={len(val_idx)})'],
        [TRAIN_COL, VAL_COL]):
    hb = ax.hexbin(yt_s, yp_s, gridsize=45, cmap=cmap, mincnt=1, linewidths=0.2)
    cb = figc.colorbar(hb, ax=ax, pad=0.02, shrink=0.85)
    cb.set_label('Point density', fontsize=10, fontweight='bold')
    cb.ax.tick_params(labelsize=9)
    for t in cb.ax.get_yticklabels(): t.set_fontweight('bold')
    ax.plot(xl, xl, color=GOOD_COL, lw=1.6, ls='--', label='Ideal y=x')
    ax.plot(xl, fv, color='#333333', lw=1.4, label=f'Fit y={rc[0]:.3f}x{rc[1]:+.3f}')
    ax.fill_between(xl, fv - rsd_gl, fv + rsd_gl, alpha=0.10, color='#333333', label='±1σ')
    if len(yt_s) > 5:
        rs_, _ = pearsonr(yt_s, yp_s); r2s_ = float(rs_ ** 2)
        rms_ = float(np.sqrt(np.mean((yt_s - yp_s) ** 2)))
    else:
        r2s_, rms_ = r2_all, rmse_gl
    ax.text(0.05, 0.95, f'R²={r2s_:.4f}\nRMSE={rms_:.4f}',
            transform=ax.transAxes, fontsize=10.5, va='top', fontweight='bold',
            bbox=dict(facecolor='white', edgecolor='#aaaaaa', boxstyle='round,pad=0.35', alpha=0.95))
    ax.set_xlim(dmin, dmax); ax.set_ylim(dmin, dmax)
    ax.set_xlabel('Original Absorbance', fontsize=12.5, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold', color=title_col, pad=6)
    ax.tick_params(labelsize=11); bold_ticks(ax)
    leg=ax.legend(fontsize=9, loc='best', handlelength=1.1, framealpha=0.92)
    for t in leg.get_texts(): t.set_fontweight('bold')
axc_c1.set_ylabel('Reconstructed Absorbance', fontsize=12.5, fontweight='bold')
axc_c1.text(-0.20, 1.10, '(c)', transform=axc_c1.transAxes,
            fontsize=15, fontweight='bold', color='#111111')

# ── (d) bottom-right: global residual scatter ────────
axc_d = figc.add_subplot(gs_outer[1, 1])
for i in range(0, len(X), stride2):
    c = TRAIN_COL if i in train_idx else VAL_COL
    axc_d.scatter(X[i, ::step], X[i, ::step] - X_recon[i, ::step],   # true - pred
                  alpha=0.06, s=1.0, color=c, rasterized=True)
axc_d.axhline(0, color=GOOD_COL, lw=1.6, ls='--', label='Zero (ideal)')
for ns, al, lb in [(1, 0.14, '±1σ'), (2, 0.06, '±2σ')]:
    axc_d.axhspan(-ns * rsd_gl, ns * rsd_gl, color=RES_COL, alpha=al, label=lb)
axc_d.plot(xl, np.poly1d(rc_res)(xl), color=RES_COL, lw=1.8, label=f'Trend={rc_res[0]:.4f}')
leg_d = [Line2D([0], [0], marker='o', color='w', markerfacecolor=TRAIN_COL, markersize=8,
                label=f'Train ({len(train_idx)})'),
         Line2D([0], [0], marker='o', color='w', markerfacecolor=VAL_COL, markersize=8,
                label=f'Val ({len(val_idx)})')]
h_d, _ = axc_d.get_legend_handles_labels()
axc_d.set_xlabel('Original Absorbance', fontsize=13, fontweight='bold')
axc_d.set_ylabel('Residual', fontsize=13, fontweight='bold')
axc_d.tick_params(labelsize=12); bold_ticks(axc_d)
axc_d.set_title('Residual Analysis', fontsize=13, fontweight='bold', pad=6)
leg=axc_d.legend(handles=leg_d + h_d, fontsize=9.5, loc='best', framealpha=0.92)
for t in leg.get_texts(): t.set_fontweight('bold')
axc_d.text(-0.08, 1.05, '(d)', transform=axc_d.transAxes,
           fontsize=15, fontweight='bold', color='#111111')

save_figure(figc, 'fig11_combined_4panel_summary')

# ════════════════════════════════════════════════════════
#  SUMMARY
# ════════════════════════════════════════════════════════
section_title("FULL PIPELINE SUMMARY")
print(f"  ── Step 1: Individual spectra ──────────────────────")
print(f"  Single CSVs found      : {len(csv_files_single)}")
print(f"  PAH skipped            : {len(skip_pah)}")
print(f"  Converted T→A          : {n_converted}")
print(f"  Already absorbance     : {n_already_A}")
print(f"  Loaded for mixing      : {len(raw_spectra)}")
print(f"\n  ── Step 2: Mixing ──────────────────────────────────")
print(f"  Ternary combos made    : {mix3_count}  (saved to {OUTPUT_MIX}/)")
print(f"  Skipped (no overlap)   : {skip_nooverlap}")
print(f"\n  ── Steps 3–8: CNN Autoencoder ──────────────────────")
print(f"  Mixed spectra loaded   : {len(X)}")
print(f"  Train / Val            : {len(train_idx)} / {len(val_idx)}")
print(f"  Overall R²             : {r2_all:.4f}")
print(f"  Overall RMSE           : {rmse_gl:.4f}")
print(f"  R² Mean / Median       : {r2_arr.mean():.4f} / {pct50:.4f}")
print(f"  Good / Fair / Poor     : {n_good}/{n_fair}/{n_poor} "
      f"({100*n_good//max(nt,1)}%/{100*n_fair//max(nt,1)}%/{100*n_poor//max(nt,1)}%)")
print(f"\n  All figures → {FIGURES_DIR}/")
print(f"  Residual convention: (true - pred) everywhere in this script.")
print(f"  Legend placement: loc='best' everywhere, adapts per spectrum.")
print(f"  All text (titles, axis labels, tick numbers, legends, annotations) is bold.")
print(f"  FIG COMBINED (fig11) tick/label/legend text sizes were increased for readability.")
print("═"*60 + "\n  Done.")

I0000 00:00:1789896646.201022   23064 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789896646.202209   23064 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789896646.379407   23064 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789896650.477437   23064 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

  INPUT_DIR   = /home/rohan/Downloads/Test Astro Molecules
  OUTPUT_MIX  = /home/rohan/Downloads/Test Astro Molecules/mixed_spectra_output_3molecules_a
  FIGURES_DIR = /home/rohan/Downloads/Test Astro Molecules/mixed_spectra_output_3molecules_a/figures_ternary


════════════════════════════════════════════════════════════
  STEP 1 — LOADING & CONVERTING INDIVIDUAL SPECTRA
════════════════════════════════════════════════════════════
  Found 253 CSV files in /home/rohan/Downloads/Test Astro Molecules
    [CONVERTED %T→A]  H2O-CH3OH-CO-NH3 = 100.10.1.1-100K.csv  raw_max=100.21  A_range=[-0.000,0.693]
    [CONVERTED %T→A]  H2O-CH3OH-CO-NH3 = 100.10.1.1-10K.csv  raw_max=95.29  A_range=[0.021,0.607]
    [CONVERTED %T→A]  H2O-CH3OH-CO-NH3 = 100.10.1.1-120K.csv  raw_max=100.20  A_range=[-0.000,0.719]
    [CONVERTED %T→A]  H2O-CH3OH-CO-NH3 = 100.10.1.1-140K.csv  raw_max=100.30  A_range=[-0.000,0.910]
    [CONVERTED %T→A]  H2O-CH3OH-CO-NH3 = 100.10.1.1-40K.csv  raw_max=107.00  A_range=[-0.000,0.

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 996, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 996, 16)        │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 996, 16)        │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 498, 16)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 498, 32)        │         2,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 498, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 249, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 249, 64)        │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 249, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 15936)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent (Dense)                  │ (None, 64)             │     1,019,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 15936)          │     1,035,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 249, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 249, 64)        │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 249, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling1d (UpSampling1D)    │ (None, 498, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 498, 32)        │        10,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 498, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling1d_1 (UpSampling1D)  │ (None, 996, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 996, 16)        │         3,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 996, 16)        │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (None, 996, 1)         │            4

 Total params: 2,091,905 (7.98 MB)

 Trainable params: 2,091,457 (7.98 MB)

 Non-trainable params: 448 (1.75 KB)


════════════════════════════════════════════════════════════
  STEP 7 — TRAIN
════════════════════════════════════════════════════════════
Epoch 1/150


E0000 00:00:1789896778.336964   23064 util.cc:131] oneDNN supports DT_INT32 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


71/71 ━━━━━━━━━━━━━━━━━━━━ 32s 254ms/step - loss: 0.2769 - mae: 0.2807 - val_loss: 0.8527 - val_mae: 0.6017 - learning_rate: 5.0000e-04
Epoch 2/150
71/71 ━━━━━━━━━━━━━━━━━━━━ 16s 225ms/step - loss: 0.0529 - mae: 0.1336 - val_loss: 0.6646 - val_mae: 0.5115 - learning_rate: 5.0000e-04
Epoch 3/150
71/71 ━━━━━━━━━━━━━━━━━━━━ 15s 207ms/step - loss: 0.0304 - mae: 0.1048 - val_loss: 0.4241 - val_mae: 0.4033 - learning_rate: 5.0000e-04
Epoch 4/150
71/71 ━━━━━━━━━━━━━━━━━━━━ 16s 223ms/step - loss: 0.0205 - mae: 0.0877 - val_loss: 0.2397 - val_mae: 0.3012 - learning_rate: 5.0000e-04
Epoch 5/150
71/71 ━━━━━━━━━━━━━━━━━━━━ 16s 224ms/step - loss: 0.0158 - mae: 0.0779 - val_loss: 0.1201 - val_mae: 0.2059 - learning_rate: 5.0000e-04
Epoch 6/150
71/71 ━━━━━━━━━━━━━━━━━━━━ 15s 205ms/step - loss: 0.0133 - mae: 0.0718 - val_loss: 0.0464 - val_mae: 0.1224 - learning_rate: 5.0000e-04
Epoch 7/150
71/71 ━━━━━━━━━━━━━━━━━━━━ 16s 224ms/step - loss: 0.0113 - mae: 0.0668 - val_loss: 0.0242 - val_mae: 0.0927 - le